In [1]:
"""
2026-03-30 jaeeun.kim
파이썬으로 네이버 블로그 검색 상위 3개의 블로그 추려서 새로운 블로그 글 구성하기
"""

import os
from dotenv import load_dotenv
import requests
from bs4 import BeautifulSoup
from openai import OpenAI
from IPython.display import display, Markdown

load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')
openai = OpenAI()

In [ ]:
# 네이버 검색 상위 3개 링크 가져오기
def get_naver_blog_urls(keyword):
    search_url = f"https://search.naver.com/search.naver?ssc=tab.blog.all&sm=tab_jum&query={keyword}"
    headers = {"User-Agent": "Mozilla/5.0"}
    
    response = requests.get(search_url, headers=headers)
    soup = BeautifulSoup(response.text, 'html.parser')

    blog_links = soup.select('a')

    links=[]
    
    for a in blog_links:
        href = a.get('href', '')
        
        # 블로그 주소 형식을 포함하고 있는지 확인
        if "blog.naver.com" in href and "/ad" not in href: # 광고 제외
            if href not in links: # 중복 제거
                links.append(href)
        
        if len(links) >= 3: # 상위 3개만 필요할 때
            break
            
    return links

In [ ]:
get_naver_blog_urls('소점')

In [16]:
# 블로그 실제 본문 텍스트 추출
def extract_blog_content(url):
    headers = {"User-Agent": "Mozilla/5.0"}
    res = requests.get(url, headers=headers)
    soup = BeautifulSoup(res.text, 'html.parser')
    
    # 네이버 블로그는 iframe 안에 실제 내용이 있음
    iframe = soup.find('iframe', id='mainFrame')
    if not iframe: return ""
    
    inner_url = "https://blog.naver.com" + iframe['src']
    inner_res = requests.get(inner_url, headers=headers)
    inner_soup = BeautifulSoup(inner_res.text, 'html.parser')
    
    # 본문 텍스트 추출 (주요 클래스명 기준)
    content = inner_soup.find('div', class_='se-main-container')
    return content.get_text(separator=' ', strip=True) if content else ""

In [26]:
# LLM을 이용한 SEO 포스팅 생성
def generate_blog_post(contents, ordered_items):
    # System Prompt: 10년차 SEO 전문가 페르소나
    brochure_system_prompt = """
    당신은 10년차 베테랑 SEO 마케팅 블로거입니다. 
    당신의 목표는 제공된 3개의 블로그 정보를 조합하여 네이버 검색 결과 최상단에 노출될 수 있는 매력적인 포스팅을 작성하는 것입니다.

    [SEO 최적화 가이드라인]
    1. 제목은 클릭을 부르는 키워드 중심(지역명 + 맛집 + 특징)으로 작성할 것.
    2. 문장 사이사이에 자연스러운 연결어(예: '뿐만 아니라', '이어서 소개할 곳은')를 사용할 것.
    3. 독자가 궁금해할 만한 구체적인 수치(역에서 도보 5분 등)를 포함할 것.
    4. 가독성을 위해 소제목([ ])과 불렛포인트를 적절히 섞을 것.
    5. 인위적인 중복보다는 풍부한 어휘를 사용하여 맥락적 연관성을 높일 것.

    [포스팅 구조 지침]
    1. 위치설명: 가는 길, 역과의 거리, 매장 외관 특징 기술.
    2. 내부 모습: 좌석 수, 인테리어 분위기, 단체석 여부 설명.
    3. 메뉴판: 주요 메뉴 구성과 특징 요약.
    4. 시킨 음식: 사용자가 제공한 파라미터를 기반으로 작성.
    5. 맛 설명: 3개 블로그의 표현을 참고하여 생생하고 차별화된 맛 리뷰 작성.
    6. 말투: -했어요, -습니다를 적절히 섞어서 친근하면서도 편안한 말투를 사용하는데, 20대 여자의 느낌이 살도록 구성해줬으면 좋겠어.
    """

    # User Prompt: 데이터 전달
    combined_context = "\n\n".join([f"[참고글 {i+1}]: {c}" for i, c in enumerate(contents)])
    user_prompt = f"""
    아래 3개 블로그의 내용을 바탕으로 새로운 블로그 포스팅을 작성해줘. 
    내가 이번에 실제로 주문한 음식은 '{ordered_items}'이야. 이 메뉴에 더 집중해서 리뷰해줘.

    {combined_context}
    """

    response = openai.chat.completions.create(
        model="gpt-4o",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": user_prompt}
        ],
        temperature=0.7
    )
    return response.choices[0].message.content

In [22]:
# 메인 파이프라인 함수
def run_blog_generator(keyword, food_list):
    print(f"🔍 '{keyword}' 관련 블로그를 탐색 중입니다...")
    urls = get_naver_blog_urls(keyword)
    
    all_contents = []
    for url in urls:
        print(f"📖 데이터 수집 중: {url}")
        all_contents.append(extract_blog_content(url))
    
    print("✍️ SEO 최적화 포스팅 생성 중...")
    final_post = generate_blog_post(all_contents, food_list)
    
    print("-" * 50)
    display(Markdown(final_post))


In [ ]:

# --- 실행부 ---
# 검색 키워드와 내가 먹은 음식을 입력하세요.
run_blog_generator("소점", "히로시마풍 오꼬노미야끼와 타코야끼")